# Outline for things to do

## 1. Loading documents for each dataset into Pinecone
- Use dataset name as metadata -> Add 1 column besides the query so that we can search by metadata -> Faster and more accurate

## 2. Create a dataframe of query:
- Columns: query_id, query, dataset_name

## 3. Things to test:

### 3.1 Simple RAG submission:
- Use raw vectore store to retrieve the top-10 most relevant documents to the query

### 3.2 Vector Search + RAG Fusion

### 3.3 Hybrid search + Reranking



In [1]:
import os
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from financerag.task import *
from financerag.retrieval import DenseRetrieval
from langchain_text_splitters.base import TextSplitter
pd.set_option('display.max_colwidth', 400)

In [2]:
load_dotenv(".env")
GG_API_KEY = os.environ.get('GOOGLE_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
LANGSMITH_API_KEY = os.environ.get('LANGSMITH_API_KEY')

In [3]:
# Set up vector database
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004")
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [3]:
# Get dataset name first
import os
dataset_names = []
for f in os.listdir('finance_dataset'):
    if f.endswith("tsv"):
       dataset_names.append(f.split('_')[0])
dataset_names  

['MultiHeirtt',
 'FinQA',
 'FinanceBench',
 'ConvFinQA',
 'FinQABench',
 'TATQA',
 'FinDER']

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=300)

In [8]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load()
    {task_variable}_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = {task_variable}.metadata.dataset_name)
    {task_variable}_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = {task_variable}.corpus, saved_index = True)
    {task_variable}.retrieve(retriever = {task_variable}_retriever, top_k = 100)
    {task_variable}.save_retrieved_results()
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load()
    multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
    multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = True)
    multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever, top_k = 100)
    multiheirtt_task.save_retrieved_results()
    

    # FinQA Task
    print(f"FinQA task")
    finqa_task = FinQATask()
    finqa_task.load()
    finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
    finqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finqa_task.corpus, saved_index = True)
    finqa_task.retrieve(retriever = finqa_task_retriever, top_k = 100)
    finqa_task.save_retrieved_results()
    

    #

In [15]:
%%time
# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = financebench_task.metadata.dataset_name)
financebench_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = financebench_task.corpus, saved_index = True)
financebench_task.retrieve(retriever = financebench_task_retriever, top_k = 100)
financebench_task.save_retrieved_results()


FinanceBench task


Loading document:: 100%|██████████| 180/180 [00:00<00:00, 7132.09it/s]


Successfully saved index of vector store to path : faiss_index/financebench_index


Retrieving result::   1%|          | 1/150 [00:00<00:50,  2.96it/s]

Len key of docs dict :2


...

Len key of docs dict :2


Retrieving result::  22%|██▏       | 33/150 [00:12<00:41,  2.81it/s]

Len key of docs dict :3


Retrieving result::  22%|██▏       | 33/150 [00:12<00:45,  2.59it/s]


KeyboardInterrupt: 

In [ ]:
from sentence_transformers import CrossEncoder

In [8]:
# 1. Load a pretrained CrossEncoder model
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", device = 'cpu')

# The texts for which to predict similarity scores
query = "How many people live in Berlin?"
passages = [
    "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.",
    "Berlin has a yearly total of about 135 million day visitors, making it one of the most-visited cities in the European Union.",
    "In 2013 around 600,000 Berliners were registered in one of the more than 2,300 sport and fitness clubs.",
]

# 2a. Either predict scores pairs of texts
scores = model.predict([(query, passage) for passage in passages])
print(scores)
# => [8.607139 5.506266 6.352977]

# 2b. Or rank a list of passages for a query
ranks = model.rank(query, passages, return_documents=True)

print("Query:", query)
for rank in ranks:
    print(f"- #{rank['corpus_id']} ({rank['score']:.2f}): {rank['text']}")

AttributeError: module 'torch' has no attribute 'get_default_device'

In [11]:
!pip install --upgrade pip

  Using cached pip-25.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1


In [13]:
!pip3 install torch==2.7.0

ERROR: Could not find a version that satisfies the requirement torch==2.7.0 (from versions: 1.7.1, 1.8.0, 1.8.1, 1.9.0, 1.9.1, 1.10.0, 1.10.1, 1.10.2, 1.11.0, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2)
ERROR: No matching distribution found for torch==2.7.0


In [5]:
import torch
torch.__version__

'2.2.2'

In [ ]:

# MultiHeirtt Task
print(f"MultiHeirtt task")
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = True)
multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever, top_k = 100)
multiheirtt_task.save_retrieved_results()


# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finqa_task.corpus, saved_index = True)
finqa_task.retrieve(retriever = finqa_task_retriever, top_k = 100)
finqa_task.save_retrieved_results()


# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = convfinqa_task.metadata.dataset_name)
convfinqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = convfinqa_task.corpus, saved_index = True)
convfinqa_task.retrieve(retriever = convfinqa_task_retriever, top_k = 100)
convfinqa_task.save_retrieved_results()


# FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqabench_task.metadata.dataset_name)
finqabench_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finqabench_task.corpus, saved_index = True)
finqabench_task.retrieve(retriever = finqabench_task_retriever, top_k = 100)
finqabench_task.save_retrieved_results()


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = tatqa_task.metadata.dataset_name)
tatqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = tatqa_task.corpus, saved_index = True)
tatqa_task.retrieve(retriever = tatqa_task_retriever, top_k = 100)
tatqa_task.save_retrieved_results()


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load()
finder_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finder_task.metadata.dataset_name)
finder_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finder_task.corpus, saved_index = True)
finder_task.retrieve(retriever = finder_task_retriever, top_k = 100)
finder_task.save_retrieved_results()


In [7]:
a = ['s', 'a', 'd']
' '.join(a)

's a d'

In [ ]:
# MultiHeirtt Task
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = True)
multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever)
multiheirtt_task.save_retrieved_results()



Loading document:: 100%|██████████| 10475/10475 [00:00<00:00, 42579.02it/s]


Successfully saved index of vector store to path : faiss_index/multiheirtt_index
Saved result successfully to ./financerag_result/multiheirtt_result.csv!


In [ ]:
# FinQA Task
finqa_task = FinQATask()
finqa_task.load()
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_for_searching_without_splitting(finqa_task.corpus, saved_index = True)
finqa_task.retrieve(retriever = finqa_task_retriever)
finqa_task.save_retrieved_results()




Loading document: 100%|██████████| 2789/2789 [00:00<00:00, 151775.10it/s]


Successfully saved index of vector store to path : faiss_index/finqa_index


Retrieving result:: 100%|██████████| 1147/1147 [07:21<00:00,  2.60it/s]

Saved result successfully to ./financerag_result/finqa_result.csv!


In [13]:

# FinanceBench Task
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = financebench_task.metadata.dataset_name)
financebench_task_retriever.load_corpus_for_searching(financebench_task.corpus, saved_index = True)
financebench_task.retrieve(retriever = financebench_task_retriever)
financebench_task.save_retrieved_results()




Loading document: 100%|██████████| 180/180 [00:00<00:00, 131942.45it/s]


Successfully saved index of vector store to path : faiss_index/financebench_index


Retrieving result:: 100%|██████████| 150/150 [01:02<00:00,  2.42it/s]

Saved result successfully to ./financerag_result/financebench_result.csv!


In [ ]:
# ConvFinQA Task
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = convfinqa_task.metadata.dataset_name)
convfinqa_task_retriever.load_corpus_for_searching_without_splitting(convfinqa_task.corpus, saved_index = True)
convfinqa_task.retrieve(retriever = convfinqa_task_retriever)
convfinqa_task.save_retrieved_results()


Loading document: 100%|██████████| 2066/2066 [00:00<00:00, 129481.68it/s]


Successfully saved index of vector store to path : faiss_index/convfinqa_index


Retrieving result:: 100%|██████████| 421/421 [02:37<00:00,  2.67it/s]

Saved result successfully to ./financerag_result/convfinqa_result.csv!


In [15]:

# FinQABench Task
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqabench_task.metadata.dataset_name)
finqabench_task_retriever.load_corpus_for_searching(finqabench_task.corpus, saved_index = True)
finqabench_task.retrieve(retriever = finqabench_task_retriever)
finqabench_task.save_retrieved_results()


Loading document: 100%|██████████| 92/92 [00:00<00:00, 89592.75it/s]


Successfully saved index of vector store to path : faiss_index/finqabench_index


Retrieving result:: 100%|██████████| 100/100 [00:36<00:00,  2.75it/s]

Saved result successfully to ./financerag_result/finqabench_result.csv!


In [ ]:

# TATQA Task
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = tatqa_task.metadata.dataset_name)
tatqa_task_retriever.load_corpus_for_searching_without_splitting(tatqa_task.corpus, saved_index = True)
tatqa_task.retrieve(retriever = tatqa_task_retriever)
tatqa_task.save_retrieved_results() 


Loading document: 100%|██████████| 2756/2756 [00:00<00:00, 182551.12it/s]


Successfully saved index of vector store to path : faiss_index/tatqa_index


Retrieving result:: 100%|██████████| 1663/1663 [15:14<00:00,  1.82it/s] 

Saved result successfully to ./financerag_result/tatqa_result.csv!


In [ ]:

# FinDER Task
finder_task = FinDERTask()
finder_task.load()
finder_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finder_task.metadata.dataset_name)
finder_task_retriever.load_corpus_for_searching_without_splitting(finder_task.corpus, saved_index = True)
finder_task.retrieve(retriever = finder_task_retriever)
finder_task.save_retrieved_results()

Loading document: 100%|██████████| 13862/13862 [00:00<00:00, 20056.98it/s]


Successfully saved index of vector store to path : faiss_index/finder_index


Retrieving result:: 100%|██████████| 216/216 [01:57<00:00,  1.84it/s]

Saved result successfully to ./financerag_result/finder_result.csv!


In [ ]:
1: Recusplitter(chunk_size = 1000, overlap_size = 20)
2 :SemanticSplitter()Hello how are you

1: hello,how , are you
2. hello | how are you

In [16]:
final_result = pd.DataFrame(columns = ['query_id', 'corpus_id'])

for dataset_name in dataset_names:
    df = pd.read_csv(f"financerag_result/{dataset_name.lower()}_result.csv")
    final_result = pd.concat([final_result, df], axis = 0)

final_result

,query_id,corpus_id
0,q82d4c6ec,d8177a896
1,q82d4c6ec,d8d3fbbaa
2,q82d4c6ec,d85eb42c0
3,q82d4c6ec,d8e817e40
4,q82d4c6ec,d8bb12b52
...,...,...
2155,q00218,ADBE20230512
2156,q00218,TSLA20231025
2157,q00218,CPNG20230812
2158,q00218,LIN20230185


In [17]:
pd.read_csv('submission.csv')

,query_id,corpus_id
0,q82d4c6ec,d8e404704
1,q82d4c6ec,d87914156
2,q82d4c6ec,d8a3219a8
3,q82d4c6ec,d8c22a07a
4,q82d4c6ec,d88e1c5b2
...,...,...
44538,q00218,GOOGL20231203
44539,q00218,JPM20234729
44540,q00218,V20231584
44541,q00218,JPM20237319


In [20]:
pd.read_csv('finance_dataset/sample_submission_.csv')

,query_id,corpus_id
0,qd496c6a0,dd4b92b32
1,qd496c6a0,dd4ba2a5a
2,qd496c6a0,dd4be1f98
3,qd496c6a0,dd4ba07d2
4,qd496c6a0,dd4ba02f0
...,...,...
46681,q1a741e68,d1b3afaba
46682,q1a741e68,d1b34e18e
46683,q1a741e68,d1b36065e
46684,q1a741e68,d1b33d05a


In [18]:
final_result.to_csv('submission_bm25.csv', index = False)

In [22]:
final_result.drop_duplicates(subset = ['query_id'])

,query_id,corpus_id
0,q82d4c6ec,d8e404704
10,q855a35a0,d89a6ea36
20,q85384530,d8ce7fc30
30,q842c8af2,d88465f0a
40,q85451756,d8646ec7e
...,...,...
2094,q00214,BRK.A20230009
2104,q00215,BRK.A20230404
2114,q00216,BRK.A20232401
2124,q00217,BRK.A20230062


In [28]:
final_queries = pd.DataFrame(columns = ['_id', 'title', 'text'])

for dataset_name in dataset_names:
    df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_queries.jsonl/queries.jsonl", lines = True)
    final_queries = pd.concat([final_queries, df], axis = 0)

final_queries

,_id,title,text
0,q82d4c6ec,,What was the sum of Fourth Quarter without tho...
1,q855a35a0,,In which section is Interest income smaller th...
2,q85384530,,If Total Forward Hedged Revenues develops with...
3,q842c8af2,,what was the ratio of the purchase in december...
4,q85451756,,what is the highest total amount of segment in...
...,...,...,...
211,q00214,,How many distinct insurance underwriting group...
212,q00215,,What is the ticker symbol for Berkshire Hathaw...
213,q00216,,What is the largest operating segment of the B...
214,q00217,,Source of invested assets of insurance busines...


In [4]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load()
    {task_variable}_bm25_retriever = BM25_Retriever()
    {task_variable}.retrieve(retriever = {task_variable}_bm25_retriever, top_k = 10)
    {task_variable}.save_retrieved_results()
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load()
    multiheirtt_task_bm25_retriever = BM25_Retriever()
    multiheirtt_task.retrieve(retriever = multiheirtt_task_bm25_retriever, top_k = 10)
    multiheirtt_task.save_retrieved_results()
    

    # FinQA Task
    print(f"FinQA task")
    finqa_task = FinQATask()
    finqa_task.load()
    finqa_task_bm25_retriever = BM25_Retriever()
    finqa_task.retrieve(retriever = finqa_task_bm25_retriever, top_k = 10)
    finqa_task.save_retrieved_results()
    

    # FinanceBench Task
    print(f"FinanceBench task")
    financebench_task = FinanceBenchTask()
    financebench_task.load()
    financebench_task_bm25_retriever = BM25_Retriever()
    financebench_task.retrieve(retriever = financebench_task_bm25_retriever, top_k = 10)
    financebench_task.save_retrieved_results()
    

    # ConvFinQA Task
    print(f"ConvFinQA task")
    convfinqa_task = ConvFinQATask()
    co

In [5]:
from financerag.retrieval import BM25, BM25_Retriever

In [15]:
%%time
# MultiHeirtt Task
print(f"MultiHeirtt task")
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
multiheirtt_task_bm25_retriever = BM25_Retriever()
multiheirtt_task.retrieve(retriever = multiheirtt_task_bm25_retriever, top_k = 10)
multiheirtt_task.save_retrieved_results()


# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
finqa_task_bm25_retriever = BM25_Retriever()
finqa_task.retrieve(retriever = finqa_task_bm25_retriever, top_k = 10)
finqa_task.save_retrieved_results()


# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_bm25_retriever = BM25_Retriever()
financebench_task.retrieve(retriever = financebench_task_bm25_retriever, top_k = 10)
financebench_task.save_retrieved_results()


# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_bm25_retriever = BM25_Retriever()
convfinqa_task.retrieve(retriever = convfinqa_task_bm25_retriever, top_k = 10)
convfinqa_task.save_retrieved_results()


# # FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_bm25_retriever = BM25_Retriever()
finqabench_task.retrieve(retriever = finqabench_task_bm25_retriever, top_k = 10)
finqabench_task.save_retrieved_results()


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_bm25_retriever = BM25_Retriever()
tatqa_task.retrieve(retriever = tatqa_task_bm25_retriever, top_k = 10)
tatqa_task.save_retrieved_results()


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load()
finder_task_bm25_retriever = BM25_Retriever()
finder_task.retrieve(retriever = finder_task_bm25_retriever, top_k = 10)
finder_task.save_retrieved_results()

MultiHeirtt task
Saved result successfully to ./financerag_result/multiheirtt_result.csv!
FinQA task
Saved result successfully to ./financerag_result/finqa_result.csv!
FinanceBench task
Saved result successfully to ./financerag_result/financebench_result.csv!
ConvFinQA task
Saved result successfully to ./financerag_result/convfinqa_result.csv!
FinQABench task
Saved result successfully to ./financerag_result/finqabench_result.csv!
TATQA task
Saved result successfully to ./financerag_result/tatqa_result.csv!
FinDER task
Saved result successfully to ./financerag_result/finder_result.csv!
CPU times: user 3min 11s, sys: 4.49 s, total: 3min 16s
Wall time: 3min 17s


In [12]:
from nltk.tokenize import word_tokenize
from langchain_community.retrievers.bm25 import BM25Retriever
retriever = BM25Retriever.from_documents(
    [
        Document(page_content="foo"),
        Document(page_content="bar"),
        Document(page_content="world"),
        Document(page_content="hello"),
        Document(page_content="foo bar"),
    ],
    k=2,
    preprocess_func=word_tokenize,
)

result = retriever.invoke("bar")
result

[Document(metadata={}, page_content='bar'),
 Document(metadata={}, page_content='foo bar')]

In [7]:
multiheirtt_task.queries

{'q82d4c6ec': 'What was the sum of Fourth Quarter without those Fourth Quarter smaller than 0, in 2012? (in million)',
 'q855a35a0': 'In which section is Interest income smaller thanProvision for credit losses?',
 'q85384530': 'If Total Forward Hedged Revenues develops with the same growing rate in 2019, what will it reach in 2020? (in million)',
 'q842c8af2': 'what was the ratio of the purchase in december 2012 to the purchase in january 2013',
 'q85451756': 'what is the highest total amount of segment in 2015?',
 'q85737826': 'How many kinds of period is the value of Customer deposits more than the 50% of the total value of Customer deposits?',
 'q827742c4': 'How many Amount for Credit Card (a) exceed the average of Amount for Credit Card (a) in 2012?',
 'q81e448f2': 'considering the years 2015-2016 , what was the decrease observed in the expense for severance and other benefits?',
 'q82305cc4': "What's the total amount of active, iSharesETFs and Non-ETF index in 2016 in equity (in m

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/mac/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [12]:
sent = multiheirtt_task.queries['q81e448f2'].lower()

stop_words = set(stopwords.words('english'))

word_tokens = word_tokenize(sent)
word_tokens

['considering',
 'the',
 'years',
 '2015-2016',
 ',',
 'what',
 'was',
 'the',
 'decrease',
 'observed',
 'in',
 'the',
 'expense',
 'for',
 'severance',
 'and',
 'other',
 'benefits',
 '?']

TypeError: list indices must be integers or slices, not tuple

In [ ]:
im

ModuleNotFoundError: No module named 'nltk'